$$(3,8)\xrightarrow{\texttt{view}}(3,2,4)\xrightarrow{\texttt{transpose(0,1)}}(2,3,4).$$

- $X\in\mathbb R^{3\times8}$: $8=h\,d_h=2\times4$，
    - $X[t,c]=10t+c,\qquad c\in\{0,\ldots,7\}$
    - $T=3$：token 数量，$t\in\{0,1,2\}$
    - $h=2$：注意力头数，$n\in\{0,1\}$
    - $d_h=4$：每个头的通道数，$j\in\{0,1,2,3\}$

-----

- view(3,2,4)：拆开最后一根轴，view 做的是“拆轴”（unflatten）；
    - (1, 8) -> (2, 4)
    - $[T,\;h\,d_h]\longrightarrow[T,\;h,\;d_h]$
        - $(3,8)\longrightarrow(3,2,4)$
    - 原来的打包通道 \(c\) 被拆成：$c=4n+j.$
    - $\widetilde X[t,n,j]=X[t,4n+j]$
        - $n$: head index
    - 例如 $t=2,n=1,j=1$：$c=4\times1+1=5,$
        - $\widetilde X[2,1,1]=X[2,5]=25$
$$
\widetilde X.\operatorname{shape}=(3,2,4)
$$

| 轴编号 | 轴含义 | 长度 |
|---:|---|---:|
| 0 | token $t$ | 3 |
| 1 | head $n$ | 2 |
| 2 | head 内通道 $j$ | 4 |

- transpose(0,1)：交换坐标名称的位置，$[T,h,d_h]\longrightarrow[h,T,d_h]$
    - $Q[n,t,j]=\widetilde X[t,n,j]$
    - 把每个长度为 4 的行向量当成一个整体，对外面的 $3\times2$ 表格做转置
    - $\widetilde X[2,1,1]=25$，经过转置后变成：$Q[1,2,1]=25$
    - 数字 25 没变，只是它的坐标从 $(t,n,j)=(2,1,1)$ 变成：$(n,t,j)=(1,2,1).$
    - 理解高维转置最可靠的方法：追踪一个具体元素的坐标。

In [2]:
import torch

In [9]:
T = 3
h = 2
d_h = 4

# X[t, c] = 10t + c
t = torch.arange(T)[:, None]             # [3, 1]
c = torch.arange(h * d_h)[None, :]       # [1, 8]
X = 10 * t + c    
X

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [10, 11, 12, 13, 14, 15, 16, 17],
        [20, 21, 22, 23, 24, 25, 26, 27]])

In [11]:
X.view(3, -1, 4)

tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7]],

        [[10, 11, 12, 13],
         [14, 15, 16, 17]],

        [[20, 21, 22, 23],
         [24, 25, 26, 27]]])

In [12]:
X.view(3, -1, 4).transpose(0, 1)

tensor([[[ 0,  1,  2,  3],
         [10, 11, 12, 13],
         [20, 21, 22, 23]],

        [[ 4,  5,  6,  7],
         [14, 15, 16, 17],
         [24, 25, 26, 27]]])

In [16]:
X.view(3, -1, 4).stride(), X.view(3, -1, 4).transpose(0, 1).stride()

((8, 4, 1), (4, 8, 1))

In [17]:
X.view(3, -1, 4).is_contiguous()

True

In [18]:
X.view(3, -1, 4).transpose(0, 1).is_contiguous()

False

### 为什么图形看起来像“重新排列”了

- 转置前是“先选 token，再选 head”：
    - `X_tilde[t, n, :]`
    - 所以图中按 $t=0,1,2$ 分成三块，每块包含两个 head。
- 转置后是“先选 head，再选 token”：
    - `Q[n, t, :]`
    - 所以图中按 $n=0,1$ 分成两块，每块包含三个 token：

```
n = 0:
    t = 0: [ 0,  1,  2,  3]
    t = 1: [10, 11, 12, 13]
    t = 2: [20, 21, 22, 23]

n = 1:
    t = 0: [ 4,  5,  6,  7]
    t = 1: [14, 15, 16, 17]
    t = 2: [24, 25, 26, 27]
```

- stride 为什么从 (8,4,1) 变成 (4,8,1)
    - 步长（stride）表示：某个索引增加 1 时，要在底层存储中前进多少个元素。
        - $\widetilde X[t,n,j],$：(3, 2, 4)
            - $\operatorname{stride}(\widetilde X)=(8,4,1)$
            - $\operatorname{offset}=8t+4n+j.$
            - $t$ 增加 1：内存前进 8 格；
            - $n$ 增加 1：内存前进 4 格；
            - $j$ 增加 1：内存前进 1 格。
    - transpose(0,1) 本质上同步做了两件事：
        - shape:  (3, 2, 4) → (2, 3, 4)
        - stride: (8, 4, 1) → (4, 8, 1)
- 为什么转置以后“非连续”
    - 对于 shape 为 $(2,3,4)$ 的普通连续张量，预期 stride 是：
        - $(12,4,1).$
    - 但 $Q$ 的实际 stride 是：$(4,8,1).$
    - 按照 $Q[n,t,j]$ 的逻辑顺序读取时，底层内存访问顺序是跳跃的：
    - 所以它是非连续张量（non-contiguous tensor）。

### view / reshape / transpose

In [23]:
X_tilde = X.view(3, 2, 4)
Q = X_tilde.transpose(0, 1)
Q.shape, Q.stride(), Q.is_contiguous()

(torch.Size([2, 3, 4]), (4, 8, 1), False)

In [24]:
Q.view(6, 4)

RuntimeError: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.

In [25]:
Q.reshape(6, 4)

tensor([[ 0,  1,  2,  3],
        [10, 11, 12, 13],
        [20, 21, 22, 23],
        [ 4,  5,  6,  7],
        [14, 15, 16, 17],
        [24, 25, 26, 27]])

| 操作 | 能共享原内存时 | 不能共享原内存时 |
|---|---|---|
| `view` | 返回共享存储的视图（view） | 直接报错 |
| `reshape` | 通常返回共享存储的视图 | 自动复制数据后返回新 tensor |


$$
\texttt{view(shape)}=\text{只在无需复制数据时改变 shape；否则报错}
$$
$$
\texttt{reshape}=\text{尽量零复制，必要时自动复制}
$$

- 使用 view，主要是为了明确表达：这里只重新解释 shape，绝不复制或移动数据。
- 默认连续 tensor 经过普通的 .view(new_shape) 后，仍然连续。
- PyTorch 官方也将 transpose() 列为产生非连续 view 的典型操作；调用 .contiguous() 才会在必要时复制成连续布局。
    - 交换两根不同且长度均大于 1 的轴，通常会变成非连续